# Masked-Augmentation Mitigation — Colab GPU runner

Fine-tunes the CheXpert-pretrained DenseNet-121 with the shortcut (border) region masked, plus a no-masking **control**, then re-runs the AUROC + out-of-lung-localization (OLL) screens on pretrained / control / masked.

**Before running:** `Runtime → Change runtime type → GPU`.

**You need the data zip** (`chexpert_data.zip`, shared on Google Drive) — the images are gitignored, so they are not in the repo. Edit `DATA_ZIP` in cell 3 to point at where it lives in your Drive.

**What to send back:** the 6 CSVs written to `results/` (`mitigation_{pretrained,control,masked}_{auroc,oll}.csv`). Cell 7 zips them for download.

**Win condition:** the *masked* model shows lower OLL than both pretrained and control while its AUROC holds. A flat/null result is fine too — we report it honestly and the paper still ships on the diagnosis.

### 1. Clone the repo + install deps

In [2]:
!git clone -b feat/mitigation https://github.com/su-andrew/cs229-shortcut-detection.git
%cd cs229-shortcut-detection
!pip install -q torch torchvision torchxrayvision grad-cam scikit-image scikit-learn pandas numpy matplotlib pyyaml

Cloning into 'cs229-shortcut-detection'...
remote: Enumerating objects: 242, done.
remote: Counting objects: 100% (242/242), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 242 (delta 140), reused 171 (delta 92), pack-reused 0 (from 0)
Receiving objects: 100% (242/242), 97.37 KiB | 890.00 KiB/s, done.
Resolving deltas: 100% (140/140), done.
/content/cs229-shortcut-detection
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 77.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 75.3 MB/s eta 0:00:00


### 2. Confirm the GPU is on

In [4]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime → Change runtime type → GPU and re-run.'
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


### 3. Mount Drive + unzip the data
Edit `DATA_ZIP` to the path of `chexpert_data.zip` in your Drive.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

# <<< EDIT THIS to where you put the data zip in Drive >>>
DATA_ZIP = '/content/drive/MyDrive/chexpert_data.zip'

!unzip -q "$DATA_ZIP" -d data/
!ls data/chexpert   # expect: PNG_train  PNG_valid  metadata_train.csv  metadata.csv

Mounted at /content/drive
metadata.csv  metadata_train.csv  PNG_train  PNG_valid


**Checkpoint:** confirm the data unzipped to the layout the scripts expect before training.

In [6]:
import os
need = ['data/chexpert/PNG_train', 'data/chexpert/PNG_valid',
        'data/chexpert/metadata_train.csv', 'data/chexpert/metadata.csv']
missing = [p for p in need if not os.path.exists(p)]
assert not missing, (
    'Data not in the expected layout. Missing: ' + str(missing) +
    '. The zip should expand to data/chexpert/... — check DATA_ZIP / re-unzip.')
print('data layout OK:', need)

data layout OK: ['data/chexpert/PNG_train', 'data/chexpert/PNG_valid', 'data/chexpert/metadata_train.csv', 'data/chexpert/metadata.csv']


### 4. Train the two models (~minutes each on GPU)
`control` = plain fine-tune (no masking); `masked` = border masked on 50% of images.

In [8]:
!python -m src.finetune --out checkpoints/control.pt --mask-frac 0.0 --epochs 4 --batch-size 16 --device cuda
!python -m src.finetune --out checkpoints/masked.pt  --mask-frac 0.5 --mask-kind border --epochs 4 --batch-size 16 --device cuda

epoch 1/4  mean_loss=0.4482
epoch 2/4  mean_loss=0.3559
epoch 3/4  mean_loss=0.3188
epoch 4/4  mean_loss=0.2930
saved checkpoint -> checkpoints/control.pt
epoch 1/4  mean_loss=0.4730
epoch 2/4  mean_loss=0.3802
epoch 3/4  mean_loss=0.3436
epoch 4/4  mean_loss=0.3178
saved checkpoint -> checkpoints/masked.pt


### 5. Evaluate all three on AUROC + OLL

In [9]:
!python -m src.mitigation_eval --tag pretrained                                  --device cuda
!python -m src.mitigation_eval --tag control --checkpoint checkpoints/control.pt --device cuda
!python -m src.mitigation_eval --tag masked  --checkpoint checkpoints/masked.pt  --device cuda

[pretrained] AUROC:
           label    auroc  n_pos  n_neg
     Atelectasis      NaN     35      0
    Cardiomegaly 0.850877     19     24
   Consolidation 0.857778     15     45
           Edema 0.860248     46     21
Pleural Effusion 0.876610     63     53
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/pspnet_chestxray_best_model_4.pth -O /root/.torchxrayvision/models_data/pspnet_chestxray_best_model_4.pth`
[██████████████████████████████████████████████████]
[pretrained] OLL:
        label  oll_pos_mean  oll_pos_lo  oll_pos_hi  oll_neg_mean  oll_fp_mean  oll_fn_mean  n_pos  n_neg  n_empty_mask  n_boot  ranked
 Cardiomegaly      0.492219    0.427245    0.562493      0.714516     0.603970     0.741917     17     19             0    2000    True
Consolidation      0.655468    0.605015    0.704760      0.769219     0.756829          NaN     13     34             0    2000    True
        Edema      0.637360    0.593679    0.683336      0.7

### 6. Quick look — the three-way comparison

In [10]:
import pandas as pd, glob
for tag in ['pretrained', 'control', 'masked']:
    for kind in ['auroc', 'oll']:
        f = f'results/mitigation_{tag}_{kind}.csv'
        print(f'\n=== {tag} / {kind} ===')
        try:
            print(pd.read_csv(f).to_string(index=False))
        except FileNotFoundError:
            print('(missing)')


=== pretrained / auroc ===
           label    auroc  n_pos  n_neg
     Atelectasis      NaN     35      0
    Cardiomegaly 0.850877     19     24
   Consolidation 0.857778     15     45
           Edema 0.860248     46     21
Pleural Effusion 0.876610     63     53

=== pretrained / oll ===
        label  oll_pos_mean  oll_pos_lo  oll_pos_hi  oll_neg_mean  oll_fp_mean  oll_fn_mean  n_pos  n_neg  n_empty_mask  n_boot  ranked
 Cardiomegaly      0.492219    0.427245    0.562493      0.714516     0.603970     0.741917     17     19             0    2000    True
Consolidation      0.655468    0.605015    0.704760      0.769219     0.756829          NaN     13     34             0    2000    True
        Edema      0.637360    0.593679    0.683336      0.772862     0.730326     0.934011     44     19             0    2000    True

=== control / auroc ===
           label    auroc  n_pos  n_neg
     Atelectasis      NaN     35      0
    Cardiomegaly 0.864035     19     24
   Consolidation 

### 7. Zip the 6 CSVs to download and send to Jonathan

In [11]:
!cd results && zip -q /content/mitigation_results.zip mitigation_*_auroc.csv mitigation_*_oll.csv && echo zipped
from google.colab import files
files.download('/content/mitigation_results.zip')

zipped


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>